### Set IBCs
Import libraries

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.interpolate import UnivariateSpline

In [2]:
# HyperNet
import sys
sys.path.append('/home/zanardi/Codes/TC/HyperNet/hypernet')

from hypernet.src.general import utils as hyp_utils
thermophysics = hyp_utils.get_thermophysics_mod()
mix_mod = thermophysics.reaction_thermo.mixture
sp_th_mod = thermophysics.specie_thermo.specie_thermo

Define inputs

In [3]:
process = ['adiabatic', 'isochoric']
species = ['O', 'O2']
freestream = {
    'p': 1000.,
    'u': 3000.,
    'T': 500.,
    'X': [0.05, 0.95]
}
specie_thermo = {
    0: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'Og.csv'
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    },
    1: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O2g.csv'
            }
        },
        'thermo': {
            'model': 'RigidRotorHarmonicOscillator',
            'Tr_index': 0,
            'Tv_index': 1
        }
    }
}

Set paths

In [4]:
path = os.path.abspath(os.getcwd() + '/../ode/')
path_to_figs = path + '/figs/'
os.makedirs(path_to_figs, exist_ok=True)

Read PLATO shock solution

In [5]:
species = ['O', 'O2']
cols = ['x','X_O','X_O2','u','Th','Tv','rho','p','nd','H','Mf']
data = pd.DataFrame(data=np.loadtxt(path + '/output_shock/shock.dat'), columns=cols)

Extract primitive variables from PLATO solution

In [6]:
sp_th = {}
for index, s in specie_thermo.items():
    args = {**s, **dict(index=index, process=process)}
    sp_th_ = sp_th_mod.SpecieThermo(**args)
    sp_th[sp_th_.specie.name] = sp_th_
for name, s in sp_th.items():
    if s.specie.n_at > 1:
        molecule = name
    else:
        atom = name
mix = mix_mod.MultiComponent(sp_th)

In [7]:
def extract_prim_var(data):
    # Extract physical variables
    X = data[['X_'+k for k in species]].to_numpy()
    T = data[['Th', 'Tv']].to_numpy()
    rho = data[['rho']].to_numpy()
    u = data[['u']].to_numpy()
    p = data[['p']].to_numpy()
    # Update thermodynamics
    for s in sp_th.values():
        s.update(T)
    # Update mixture
    mix.update(X, var='X')
    # Compute needed physical variables
    Y = np.hstack([sp_th[k].specie.Y.numpy() for k in species])
    return tf.concat([Y*rho, u, T], 1).numpy()

In [8]:
prim_sol = extract_prim_var(data)

Compute freestream primitive variables

In [9]:
# Get temperatures and molar fractions
T = np.array([freestream['T']]*2).reshape(1,2)
X = np.array(freestream['X']).reshape(1,2)
# Update thermodynamics
for s in sp_th.values():
    s.update(T)
# Update mixture
mix.update(X, var='X')
# Compute needed physical variables
u = np.array(freestream['u']).reshape(1,1)
rho = mix._rho(freestream['p'], T)
Y = np.hstack([sp_th[k].specie.Y.numpy() for k in species])
# Concatenate
prim_freestream = tf.concat([Y*rho, u, T], 1).numpy()

Write IBCs to file

In [10]:
def replace(file, patterns, replaces):
    file_old = open(file, 'r')
    file_new = open(file+'_new', 'w')
    for line in file_old:
        if any([p in line for p in patterns]):
            for i, p in enumerate(patterns):
                if p in line:
                    line_new = replaces[i]
                    break
        else:
            line_new = line
        file_new.write(line_new)
    file_new.close()
    file_old.close()
    os.rename(file+'_new', file)

In [11]:
models = ['surrogate', 'thermolib']
schemes = ['implicit', 'strang']

inlet_prim = np.array2string(
    prim_freestream.squeeze(),
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')
init_sol_prim = np.array2string(
    np.hstack([prim_freestream, np.take(prim_sol, [-1], axis=0)]).squeeze(),
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')

for m in models:
    for s in schemes:
        file = f'../pde/{m}/{s}/input/inputfile.sh'
        if os.path.exists(file):
            replace(
                file=os.path.abspath(file),
                patterns=(
                    "INLET.PRIM",
                    "INIT_SOL_PRIM"
                ),
                replaces=(
                    f"INLET.PRIM = {inlet_prim[1:-1]}\n",
                    f"INIT_SOL_PRIM = {init_sol_prim[1:-1]}\n"
                )
            )